# SMS Spam Classification using Markov Chains and Bayes' Theorem

## 1. Data Processing
Load `spam.csv`, split into train (80%) and test (20%), and preprocess the messages.

In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import math

In [25]:
df = pd.read_csv("spam.csv", encoding = 'latin1')
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [26]:
df["Category"].value_counts()

,count
Category,
ham,4825
spam,747


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Message"],
    df["Category"],
    test_size = 0.2,
    random_state = 42
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 4457
Test: 1115


In [28]:
def preprocess(message):
    message = message.lower()
    message = re.sub(r'[^a-z0-9\s]', ' ', message)
    message = message.split()
    return message

X_train_tokens = X_train.apply(preprocess)
X_test_tokens = X_test.apply(preprocess)

print(X_train_tokens.head())

1978    [reply, to, win, 100, weekly, where, will, the...
3989    [hello, sort, of, out, in, town, already, that...
3935    [how, come, guoyang, go, n, tell, her, then, u...
4078    [hey, sathya, till, now, we, dint, meet, not, ...
4086    [orange, brings, you, ringtones, from, all, ti...
Name: Message, dtype: object


## 2. Training
Calculate log prior probabilities, build vocabulary, and compute initial and transition probabilities for spam and ham.

In [29]:
counts = y_train.value_counts()

spam_count = counts["spam"]
ham_count = counts["ham"]
total = len(y_train)

log_prior_spam = math.log(spam_count/total)
log_prior_ham = math.log(ham_count/total)

print("Spam:", spam_count, "Ham:", ham_count)
print("Log P(Spam):", log_prior_spam)
print("Log P(Ham):", log_prior_ham)

Spam: 598 Ham: 3859
Log P(Spam): -2.008640418995924
Log P(Ham): -0.14406781140893707


In [30]:
spam_messages = []
ham_messages = []
for tokens, label in zip(X_train_tokens, y_train):
  if label == "spam":
    spam_messages.append(tokens)
  else:
    ham_messages.append(tokens)

print("Spam messages:", len(spam_messages))  # Number of spam messages in train
print("Ham messages:", len(ham_messages))    # Number of ham messages in train

Spam messages: 598
Ham messages: 3859


In [31]:
def build_vocabulary(messages):
  vocab = set()
  for message in messages:
    for word in message:
      vocab.add(word)
  return vocab

spam_vocab = build_vocabulary(spam_messages)
ham_vocab = build_vocabulary(ham_messages)
vocab = build_vocabulary(X_train_tokens)

print("Spam vocabulary:", len(spam_vocab))
print("Ham vocabulary:", len(ham_vocab))
print("Total vocabulary:", len(vocab))

Spam vocabulary: 2607
Ham vocabulary: 6102
Total vocabulary: 7733


### Initial and Transition Counts
Count first words and word pairs separately for spam and ham messages.

In [32]:
def count_markov(messages):
  initial_counts = {}
  transition_counts = {}

  for words in messages:
    if len(words) == 0:
      continue

    # Initial: first word
    first_word = words[0]
    if first_word in initial_counts:
      initial_counts[first_word] = initial_counts[first_word] + 1
    else:
      initial_counts[first_word] = 1

    # Transitions: word pairs
    for i in range(len(words) - 1):
      prev = words[i]
      nxt = words[i + 1]

      if prev not in transition_counts:
        transition_counts[prev] = {}

      if nxt in transition_counts[prev]:
        transition_counts[prev][nxt] = transition_counts[prev][nxt] + 1
      else:
        transition_counts[prev][nxt] = 1

  return initial_counts, transition_counts


spam_initial_counts, spam_transition_counts = count_markov(spam_messages)
ham_initial_counts, ham_transition_counts = count_markov(ham_messages)

print("Spam - first words:", len(spam_initial_counts), " previous words:", len(spam_transition_counts))
print("Ham - first words:", len(ham_initial_counts), " previous words:", len(ham_transition_counts))

Spam - first words: 212  previous words: 2455
Ham - first words: 750  previous words: 5655


### Smoothed Log Probabilities
Compute initial and transition probabilities with epsilon smoothing, in log space.

In [34]:
epsilon = 1
M = len(vocab)

spam_N = sum(spam_initial_counts.values())
ham_N = sum(ham_initial_counts.values())


def log_initial_prob(word, initial_counts, N):
  count = initial_counts.get(word, 0)
  return math.log((count + epsilon) / (N + epsilon * M))


def log_transition_prob(prev, nxt, transition_counts):
  page = transition_counts.get(prev, {})
  count_prev = sum(page.values())
  count_pair = page.get(nxt, 0)
  return math.log((count_pair + epsilon) / (count_prev + epsilon * M))


print("M:", M, " spam N:", spam_N, " ham N:", ham_N)

M: 7733  spam N: 598  ham N: 3857


## 3. Prediction
For each message, calculate log P(M|spam) + log P(spam) and log P(M|ham) + log P(ham), and choose the larger one.

In [35]:
def log_likelihood(words, initial_counts, transition_counts, N):
  if len(words) == 0:
    return 0

  score = log_initial_prob(words[0], initial_counts, N)
  for i in range(len(words) - 1):
    score = score + log_transition_prob(words[i], words[i + 1], transition_counts)
  return score


def classify(words):
  spam_score = log_likelihood(words, spam_initial_counts, spam_transition_counts, spam_N) + log_prior_spam
  ham_score = log_likelihood(words, ham_initial_counts, ham_transition_counts, ham_N) + log_prior_ham

  if spam_score > ham_score:
    return "spam"
  else:
    return "ham"

## 4. Evaluation
Calculate accuracy on the test set.

In [37]:
correct = 0

for words, true_label in zip(X_test_tokens, y_test):
  predicted = classify(words)
  if predicted == true_label:
    correct = correct + 1

accuracy = correct / len(y_test)

print("Correct:", correct, "out of", len(y_test))
print("Accuracy:", accuracy)

Correct: 1089 out of 1115
Accuracy: 0.9766816143497757
